In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")
data_path = os.getenv("Data_path_featured")

In [3]:
df=pd.read_csv(data_path)
df.head()

,CustomerID,SeniorCitizen,Partner,Dependents,Tenure,InternetService,OnlineSecurity,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,0,Yes,No,1,DSL,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,0,No,No,34,DSL,Yes,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,0,No,No,2,DSL,Yes,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,0,No,No,45,DSL,Yes,Yes,One year,No,Bank transfer,42.30,1840.75,0
4,9237-HQITU,0,No,No,2,Fiber optic,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [4]:
x=df.drop(['Churn','CustomerID'],axis=1)
y=df["Churn"]

In [5]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42,stratify=y)

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler

In [8]:
x.columns

Index(['SeniorCitizen', 'Partner', 'Dependents', 'Tenure', 'InternetService',
       'OnlineSecurity', 'TechSupport', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges'],
      dtype='object')

In [9]:
binary_colm = ['SeniorCitizen','Partner','Dependents','OnlineSecurity','TechSupport','PaperlessBilling']
multi_cat_colm = ['InternetService', 'PaymentMethod', 'Contract']
numerical_colm = ['Tenure', 'MonthlyCharges', 'TotalCharges']

In [10]:
binary_pipe = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])

multi_cat_pipe = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))])

numerical_pipe = Pipeline(steps=[
    ('scaler', RobustScaler())])

In [11]:
preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_pipe, binary_colm),
    ('multi', multi_cat_pipe, multi_cat_colm),
    ('num', numerical_pipe, numerical_colm)])

In [12]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [13]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())}

In [14]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipe.fit(x_train, y_train)
    
    y_pred = pipe.predict(x_test)
    y_proba = pipe.predict_proba(x_test)[:, 1]
    
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


--- Logistic Regression ---
              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1035
           1       0.50      0.80      0.62       374

    accuracy                           0.74      1409
   macro avg       0.70      0.76      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC: 0.839
Confusion Matrix:
 [[738 297]
 [ 75 299]]

--- Random Forest ---
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.63      0.48      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.70      1409
weighted avg       0.77      0.79      0.78      1409

ROC-AUC: 0.8197
Confusion Matrix:
 [[928 107]
 [193 181]]

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.87      0.79      0.83      1035
           1       0.53      0.67      0.59       374

In [15]:
#Logistic Regression(Best for Minimizing Revenue Loss)
#XGBoost(Best Balanced Performance)
import joblib

best_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()))
])
best_pipe.fit(x_train, y_train)

model_path = os.path.join('E:\coding\PROJECTS\customer_churn_prediction\models', 'churn_model_XGB.pkl')
joblib.dump(best_pipe, model_path)

<>:11: SyntaxWarning: invalid escape sequence '\c'
<>:11: SyntaxWarning: invalid escape sequence '\c'
C:\Users\hunnu\AppData\Local\Temp\ipykernel_30168\2642789036.py:11: SyntaxWarning: invalid escape sequence '\c'
  model_path = os.path.join('E:\coding\PROJECTS\customer_churn_prediction\models', 'churn_model_XGB.pkl')


['E:\\coding\\PROJECTS\\customer_churn_prediction\\models\\churn_model_XGB.pkl']